In [ ]:
import json
import random
import requests
import threading
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import sleep, perf_counter

import pandas as pd
from loguru import logger
from scrapetube import get_search_cc
from tqdm.auto import tqdm

In [ ]:
today_date = datetime.now()
today_string = today_date.strftime("%Y-%m-%d").replace("-", "")
logger.add(f"../logs/scrapetube_cc_00_extract_data_{today_string}.log")
pd.options.display.max_colwidth = 200

In [ ]:
def test_proxy(
    proxy_config: dict[str, str], timeout: int = 10
) -> tuple[bool, float, str]:
    """
    Test if a proxy is working by making requests to multiple endpoints.

    Args:
        proxy_config: Dictionary with 'http' and 'https' proxy URLs
        timeout: Request timeout in seconds

    Returns:
        Tuple of (is_working, response_time, error_message)
    """

    test_urls = [
        # "https://httpbin.org/ip",  # Returns IP info
        # "https://www.google.com",  # Test HTTPS
        "https://www.youtube.com",  # Test YouTube (relevant for your use case)
    ]
    

    for url in test_urls:
        try:
            start_time = perf_counter()
            response = requests.get(
                url,
                proxies=proxy_config,
                timeout=timeout,
                headers={
                    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
                },
            )
            response_time = perf_counter() - start_time

            if response.status_code == 200:
                return True, response_time, "Success"

        except requests.exceptions.RequestException as e:
            continue

    return False, 0.0, "Failed all test URLs"


def test_proxy_list(
    proxy_list: list[dict[str, str]], max_workers: int = 10
) -> list[dict]:
    """
    Test multiple proxies concurrently.

    Args:
        proxy_list: List of proxy configurations
        max_workers: Number of concurrent workers

    Returns:
        List of dictionaries with test results
    """
    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all proxy tests
        future_to_proxy = {
            executor.submit(test_proxy, proxy): proxy for proxy in proxy_list
        }

        # Collect results
        for future in tqdm(
            as_completed(future_to_proxy), total=len(proxy_list), desc="Testing proxies"
        ):
            proxy = future_to_proxy[future]
            try:
                is_working, response_time, error_msg = future.result()
                results.append(
                    {
                        "proxy": proxy,
                        "working": is_working,
                        "response_time": response_time,
                        "error": error_msg,
                    }
                )
            except Exception as e:
                results.append(
                    {
                        "proxy": proxy,
                        "working": False,
                        "response_time": 0.0,
                        "error": str(e),
                    }
                )

    return results


def create_proxy_config(ip: str, port: str) -> dict[str, str]:
    """
    Create proxy configuration dictionary from IP and port.

    Args:
        ip: Proxy IP address
        port: Proxy port

    Returns:
        Proxy configuration dictionary
    """
    proxy_url = f"http://{ip}:{port}"
    return {"http": proxy_url, "https": proxy_url}


free_proxies = [
    create_proxy_config("8.210.117.141", "8888"),  # Hong Kong
    create_proxy_config("123.141.181.86", "5031"),  # South Korea
    create_proxy_config("72.10.160.170", "3949"),  # Canada
    create_proxy_config("47.236.163.74", "8080"),  # Singapore
    create_proxy_config("219.65.73.81", "80"),  # India
    create_proxy_config("57.129.81.201", "8080"),  # Germany
    create_proxy_config("59.29.182.162", "8888"),  # South Korea
    create_proxy_config("190.58.248.86", "80"),  # Trinidad and Tobago
    create_proxy_config("50.122.86.118", "80"),  # United States
    create_proxy_config("4.156.78.45", "80"),  # United States
    create_proxy_config("72.10.160.91", "18749"),  # Canada
    create_proxy_config("51.79.99.237", "4502"),  # Canada
    create_proxy_config("159.89.239.166", "18084"),  # United States
    create_proxy_config("195.158.8.123", "3128"),  # Uzbekistan
    create_proxy_config("5.78.129.53", "80"),  # United States
    create_proxy_config("37.187.74.125", "80"),  # France
    create_proxy_config("23.247.136.254", "80"),  # United States
    create_proxy_config("139.162.78.109", "8080"),  # Japan
    create_proxy_config("4.245.123.244", "80"),  # Netherlands
    create_proxy_config("92.67.186.210", "80"),  # Netherlands
    create_proxy_config("4.195.16.140", "80"),  # Australia
    create_proxy_config("78.47.127.91", "80"),  # Germany
    create_proxy_config("147.75.34.105", "443"),  # Netherlands
    create_proxy_config("59.7.246.4", "80"),  # South Korea
    create_proxy_config("108.141.130.146", "80"),  # Netherlands
    create_proxy_config("90.162.35.34", "80"),  # Spain
    create_proxy_config("129.159.38.24", "80"),  # United States
    create_proxy_config("91.132.92.150", "80"),  # Denmark
    create_proxy_config("90.156.169.163", "80"),  # Russian Federation
    create_proxy_config("123.141.181.49", "5031"),  # South Korea
    create_proxy_config("62.99.138.162", "80"),  # Austria
    create_proxy_config("189.202.188.149", "80"),  # Mexico
    create_proxy_config("80.120.49.242", "80"),  # Austria
    create_proxy_config("195.88.71.201", "8888"),  # United Kingdom
    create_proxy_config("89.58.55.33", "80"),  # Germany
    create_proxy_config("41.191.203.162", "80"),  # Lesotho
    create_proxy_config("213.143.113.82", "80"),  # Austria
    create_proxy_config("89.58.57.45", "80"),  # Germany
    create_proxy_config("152.53.168.53", "44887"),  # Austria
    create_proxy_config("91.107.149.78", "80"),  # Germany
    create_proxy_config("223.135.156.183", "8080"),  # Japan
    create_proxy_config("155.94.241.132", "3128"),  # United States
]
r = test_proxy_list(free_proxies)

In [ ]:
working_results = [result for result in r if result["working"]]
working_results.sort(key=lambda x: x["response_time"])
working_proxies = [result["proxy"] for result in working_results]

In [ ]:
working_results

In [ ]:
# Thread-safe locks for concurrent file operations (NECESSARY for ThreadPoolExecutor)
csv_lock = threading.Lock()
json_lock = threading.Lock()


def save_to_csv(df: pd.DataFrame, file_path_csv: str) -> None:
    """
    Save or append DataFrame to a CSV file in a thread-safe manner.
    
    Args:
        df: DataFrame to save
        file_path_csv: Path to the CSV file
    """
    if df.empty:
        logger.warning("Attempted to save empty DataFrame to CSV")
        return
        
    with csv_lock:
        try:
            # Check if file exists by trying to read it
            with open(file_path_csv, "r", encoding='utf-8'):
                # File exists, append without header
                df.to_csv(file_path_csv, mode="a", header=False, index=False, encoding='utf-8')
        except FileNotFoundError:
            # File doesn't exist, create with header
            df.to_csv(file_path_csv, mode="w", header=True, index=False, encoding='utf-8')
        except Exception as e:
            logger.error(f"Error saving to CSV {file_path_csv}: {e}")


def save_to_json(video_meta_list: list[dict], file_path_json: str) -> None:
    """
    Append data to a JSON file in a thread-safe manner.
    
    Args:
        video_meta_list: List of video metadata dictionaries to append
        file_path_json: Path to the JSON file
    """
    if not video_meta_list:
        logger.warning("Attempted to save empty list to JSON")
        return
        
    with json_lock:
        try:
            # Try to read existing data
            with open(file_path_json, "r", encoding='utf-8') as file:
                existing_data = json.load(file)
        except FileNotFoundError:
            existing_data = []
        except json.JSONDecodeError as e:
            logger.error(f"JSON decode error in {file_path_json}: {e}")
            existing_data = []
        except Exception as e:
            logger.error(f"Error reading JSON {file_path_json}: {e}")
            return

        # Extend existing data with new data
        existing_data.extend(video_meta_list)

        try:
            # Write updated data back to file
            with open(file_path_json, "w", encoding='utf-8') as file:
                json.dump(existing_data, file, indent=2, ensure_ascii=False)
        except Exception as e:
            logger.error(f"Error writing to JSON {file_path_json}: {e}")


class ProxyRotator:
    """Efficient proxy rotation with health tracking"""
    
    def __init__(self, proxies: list[dict[str, str]]):
        self.proxies = proxies.copy()
        self.current_index = 0
        self.failed_proxies = set()
        self.success_count = {}
        
    def get_next_proxy(self) -> dict[str, str] | None:
        """Get next available proxy, skipping failed ones"""
        if len(self.failed_proxies) >= len(self.proxies):
            # All proxies failed, reset and try again
            self.failed_proxies.clear()
            logger.warning("All proxies failed, resetting proxy rotation")
            
        available_proxies = [
            (i, proxy) for i, proxy in enumerate(self.proxies) 
            if i not in self.failed_proxies
        ]
        
        if not available_proxies:
            return None
            
        # Choose randomly from available proxies
        index, proxy = random.choice(available_proxies)
        self.current_index = index
        return proxy
    
    def mark_proxy_failed(self, proxy: dict[str, str]) -> None:
        """Mark a proxy as failed"""
        for i, p in enumerate(self.proxies):
            if p == proxy:
                self.failed_proxies.add(i)
                break
                
    def mark_proxy_success(self, proxy: dict[str, str]) -> None:
        """Mark a proxy as successful"""
        for i, p in enumerate(self.proxies):
            if p == proxy:
                self.success_count[i] = self.success_count.get(i, 0) + 1
                # Remove from failed list if it was there
                self.failed_proxies.discard(i)
                break


def extract_video_metadata(video_meta: dict) -> dict | None:
    """
    Extract and clean video metadata from raw response.
    
    Args:
        video_meta: Raw video metadata dictionary from YouTube API
        
    Returns:
        Cleaned metadata dictionary or None if extraction fails
    """
    try:
        # Extract video ID
        video_id = video_meta.get("videoId", "").strip()
        if not video_id:
            return None

        # Extract title with safe navigation
        title_data = video_meta.get("title", {})
        if isinstance(title_data, dict):
            runs = title_data.get("runs", [])
            if runs and isinstance(runs[0], dict):
                title = runs[0].get("text", "").strip()
            else:
                title = title_data.get("simpleText", "").strip()
        else:
            title = str(title_data).strip() if title_data else ""
        
        # Extract channel info with safe navigation
        byline_data = video_meta.get("longBylineText", {})
        if isinstance(byline_data, dict):
            runs = byline_data.get("runs", [])
            if runs and isinstance(runs[0], dict):
                channel = runs[0].get("text", "").strip()
                # Extract channel URL
                nav_endpoint = runs[0].get("navigationEndpoint", {})
                browse_endpoint = nav_endpoint.get("browseEndpoint", {})
                at_channel = browse_endpoint.get("canonicalBaseUrl", "").strip()
            else:
                channel = byline_data.get("simpleText", "").strip()
                at_channel = ""
        else:
            channel = ""
            at_channel = ""

        # Extract view count
        view_count_data = video_meta.get("viewCountText", {})
        if isinstance(view_count_data, dict):
            view_count = view_count_data.get("simpleText", "")
        else:
            view_count = str(view_count_data) if view_count_data else ""

        return {
            "video_id": video_id or None,
            "title": title or None,
            "channel": channel or None,
            "at_channel": at_channel or None,
            "view_count": view_count or None,
        }
        
    except Exception as e:
        logger.debug(f"Error extracting metadata: {e}")
        return None


def fetch_video_meta(
    query: str,
    limit: int,
    sleep_min: int,
    sleep_max: int,
    sp_filter: str,
    results_type: str,
    proxies: list[dict[str, str]],
    video_base_url: str,
    file_path_csv: str,
    file_path_json: str,
) -> None:
    """
    Fetch video metadata for a single query with improved proxy rotation.
    
    Args:
        query: Search query string
        limit: Maximum number of results to fetch
        sleep_min: Minimum sleep time between requests
        sleep_max: Maximum sleep time between requests
        sp_filter: YouTube search filter parameter
        results_type: Type of results to fetch (usually "video")
        proxies: List of proxy configurations
        video_base_url: Base URL for video links
        file_path_csv: Path to save CSV results
        file_path_json: Path to save JSON results
    """
    if not proxies:
        logger.error(f"No proxies provided for query '{query}'")
        return
        
    logger.info(f"🔍 Searching YouTube for: {query}")
    
    # Initialize proxy rotator
    proxy_rotator = ProxyRotator(proxies)
    max_proxy_attempts = min(len(proxies) * 2, 10)  # Limit retry attempts
    
    video_meta_list = []
    
    # Try multiple proxies with improved rotation
    for attempt in range(max_proxy_attempts):
        proxy = proxy_rotator.get_next_proxy()
        if not proxy:
            logger.error(f"No available proxies for query '{query}'")
            break
            
        try:
            sleep_time = random.randint(sleep_min, sleep_max)
            logger.debug(f"Using proxy {proxy} for query '{query}' (attempt {attempt + 1})")
            
            video_meta_generator = get_search_cc(
                query=query,
                limit=limit,
                sleep=sleep_time,
                sp_filter=sp_filter,
                results_type=results_type,
                proxies=proxy,
            )
            
            video_meta_list = list(video_meta_generator)
            
            if video_meta_list:
                proxy_rotator.mark_proxy_success(proxy)
                logger.info(f"✅ Found {len(video_meta_list)} results for '{query}'")
                break
            else:
                logger.warning(f"⚠️ Proxy returned empty results for '{query}' - may be no CC videos")
                # Don't mark as failed for empty results, might be legitimate
                break
                
        except Exception as e:
            proxy_rotator.mark_proxy_failed(proxy)
            logger.warning(f"❌ Proxy {proxy} failed for '{query}': {e}")
            
            if attempt == max_proxy_attempts - 1:
                logger.error(f"All proxy attempts failed for query '{query}'")
                return

    if not video_meta_list:
        logger.warning(f"No results found for query '{query}'")
        return

    # Process metadata with improved extraction
    items = []
    successful_extractions = 0
    
    logger.info(f"🔄 Processing {len(video_meta_list)} videos for '{query}'...")
    
    for video_meta in tqdm(
        video_meta_list,
        total=min(len(video_meta_list), limit),
        desc=f"Processing {query}",
        leave=False
    ):
        extracted_data = extract_video_metadata(video_meta)
        if extracted_data:
            # Add query and video URL
            extracted_data.update({
                "query": query,
                "video_url": f"{video_base_url}{extracted_data['video_id']}"
            })
            items.append(extracted_data)
            successful_extractions += 1

    logger.info(f"📊 Successfully extracted {successful_extractions}/{len(video_meta_list)} videos for '{query}'")

    if items:
        df = pd.DataFrame(items)
        save_to_csv(df, file_path_csv)
        save_to_json(video_meta_list, file_path_json)
    else:
        logger.warning(f"No valid items extracted for query '{query}'")


def get_cc_video_meta(
    queries: list[str],
    limit: int,
    sleep_min: int,
    sleep_max: int,
    sp_filter: str,
    file_path_csv: str,
    file_path_json: str,
    results_type: str = "video",
    proxies: list[dict[str, str]] = None,
    max_workers: int = 4,
) -> None:
    """
    Retrieve video metadata for multiple queries using parallel processing.
    
    Args:
        queries: List of search queries
        limit: Maximum results per query
        sleep_min: Minimum sleep time between requests
        sleep_max: Maximum sleep time between requests
        sp_filter: YouTube search filter
        file_path_csv: Path to save CSV results
        file_path_json: Path to save JSON results
        results_type: Type of results to fetch
        proxies: List of proxy configurations
        max_workers: Number of parallel workers
    """
    if not queries:
        logger.error("No queries provided")
        return
        
    if not proxies:
        logger.error("No proxies provided")
        return
        
    video_base_url = "https://www.youtube.com/watch?v="
    
    logger.info(f"🚀 Starting metadata fetch for {len(queries)} queries using {max_workers} workers")
    logger.info(f"📁 Results will be saved to: {file_path_csv} and {file_path_json}")

    successful_queries = 0
    failed_queries = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_query = {
            executor.submit(
                fetch_video_meta,
                query,
                limit,
                sleep_min,
                sleep_max,
                sp_filter,
                results_type,
                proxies,
                video_base_url,
                file_path_csv,
                file_path_json,
            ): query
            for query in queries
        }
        
        # Process completed tasks
        for future in tqdm(
            as_completed(future_to_query), 
            total=len(queries), 
            desc="Processing queries"
        ):
            query = future_to_query[future]
            try:
                future.result()  # This will raise an exception if the task failed
                successful_queries += 1
            except Exception as e:
                failed_queries += 1
                logger.error(f"❌ Error processing query '{query}': {e}")

    logger.info(f"✅ Finished! {successful_queries} successful, {failed_queries} failed queries")
    logger.info(f"📊 Results saved to {file_path_csv} and {file_path_json}")

## Use query string to get relevant channels

In [ ]:
query_strings = [
    "คณิตศาสตร์",
    "ฟิสิกส์",
    "เคมี",
    "ชีววิทยา",
    "สังคมศาสตร์",
    "ประวัติศาสตร์",
    "การงานอาชีพ",
    "พลศึกษา",
    "ภาษาไทย",
    "สุขศึกษา",
    "การเมือง",
    "การท่องเที่ยว",
    "กฎหมาย",
    "วิศวกรรมศาสตร์",
    "คอมพิวเตอร์",
    "การเขียนโปรแกรม",
    "การใช้ชีวิต",
    "จิตวิทยา",
    "การเกษตร",
    "การแพทย์",
    "การรักษาโรค",
    "สถาปัตยกรรม",
    "การบริหารธุรกิจ",
    "การประมง",
    "การศึกษา",
    "อุตสหกรรม",
    "สิ่งแวดล้อม",
    "การพยาบาล",
    "กีฬา",
    "การโรงแรม",
    "สาธารณสุข",
    "ทรัพยากรธรรมชาติ",
    "เศรษฐศาสตร์",
    "การเงิน",
    "มหาวิทยาลัย",
    "การพัฒนาตนเอง",  # Self-improvement
    "ดาราศาสตร์",  # Astronomy
    "การถ่ายภาพ",  # Photography
    "การออกแบบกราฟิก",  # Graphic Design
    "ภาษาอังกฤษ",  # English Language
    "วัฒนธรรม",  # Culture
    "เทคโนโลยี",  # Technology
    "การทำอาหาร",  # Cooking
    "ดนตรี",  # Music
    "การเงินส่วนบุคคล"  # Personal Finance
    # Trending or Popular Content
    "เกมออนไลน์",  # Online Games
    "การลงทุน",  # Investing
    "สุขภาพจิต",  # Mental Health
    "ปัญญาประดิษฐ์",  # Artificial Intelligence
    "การเรียนรู้ของเครื่อง",  # Machine Learning
    # Lifestyle and Personal Development
    "การออกกำลังกาย",  # Fitness
    "โยคะ",  # Yoga
    "การวางแผนชีวิต",  # Life Planning
    "มังสวิรัติ",  # Vegetarianism
    "การจัดการเวลา",  # Time Management
    # Creative and DIY
    "งานฝีมือ",  # Handicrafts
    "การออกแบบภายใน",  # Interior Design
    "การทำสวน",  # Gardening
    "การเขียนนิยาย",  # Novel Writing
    "การซ่อมแซมบ้าน",  # Home Repairs
    # Educational and Technical
    "หุ่นยนต์",  # Robotics
    "การเขียนโค้ด",  # Coding
    "ข้อมูลขนาดใหญ่",  # Big Data
    "การสอนออนไลน์",  # Online Teaching
    "การวิเคราะห์ข้อมูล",  # Data Analysis
    # Cultural and Entertainment
    "ภาพยนตร์",  # Movies
    "อนิเมะ",  # Anime
    "การเดินทางท่องเที่ยว",  # Travel Adventures
    "เพลงไทย",  # Thai Music
    "การเรียนภาษาใหม่",  # Learning a New Language
]
limit = 1_000
sleep_min = 5
sleep_max = 15
sp_filter = "relevance"
workers = 128
file_path_csv = f"query_string_df_{today_string}.csv"
file_path_json = f"query_string_df_{today_string}.json"

In [ ]:
get_cc_video_meta(
    queries=query_strings,
    limit=limit,
    sleep_min=sleep_min,
    sleep_max=sleep_max,
    sp_filter=sp_filter,
    file_path_csv=file_path_csv,
    file_path_json=file_path_json,
    max_workers=workers,
    proxies=proxies,
)

In [ ]:
1

## Get CC videos in each channel

In [ ]:
import pandas as pd

query_string_df = pd.read_csv("../data/query_string_df_20241211.csv")
query_string_df.info()

In [ ]:
query_string_df

In [ ]:
query_string_df["channel"].nunique()

In [ ]:
## use this to continue download: we just filter out already downloaded
## if load from scratch, skip this cell
# filter = pd.read_csv("./data/channel_df_20241211.csv")
# print(filter.shape)
# downloaded_channel = filter["channel"].unique().tolist()
# len(downloaded_channel)

# channels = query_string_df["channel"].unique().tolist()
# print(len(channels))
# channels = [c for c in channels if c not in downloaded_channel]
# print(len(channels))

In [ ]:
channels = ('"' + query_string_df["channel"] + '"').unique().tolist()
channels, len(channels)

In [ ]:
# # Set up logging
# logging.basicConfig(
#     level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
# )


# def get_cc_video_meta(
#     queries: List[str],
#     limit: int,
#     sleep_min: int,
#     sleep_max: int,
#     sp_filter: str,
#     results_type: str = "video",
#     proxies: Optional[Dict[str, str]] = None,
# ) -> Tuple[pd.DataFrame, List[dict]]:
#     """
#     Retrieve video metadata for a list of queries using Scrapetube.

#     Args:
#         queries (List[str]): List of search queries to fetch metadata for.
#         limit (int): Maximum number of results per query.
#         sleep_min (int): Minimum sleep time between requests.
#         sleep_max (int): Maximum sleep time between requests.
#         sp_filter (str): Filter parameter for YouTube search.
#         results_type (str, optional): Type of result to fetch (default is 'video').
#         proxies (Optional[Dict[str, str]], optional): Proxy configuration (default is None).

#     Returns:
#         Tuple[List[pd.DataFrame], List[dict]]: A tuple containing:
#             - A list of Pandas DataFrames with metadata.
#             - A list of raw JSON responses.
#     """
#     video_base_url = "https://www.youtube.com/watch?v="
#     all_dataframes = []
#     all_jsons = []

#     for query in tqdm(queries, total=len(queries)):
#         logging.info(f"Searching YouTube for: {query}")

#         # Random sleep time for throttling
#         sleep_time = random.randint(sleep_min, sleep_max)
#         try:
#             video_meta_generator = get_search_cc(
#                 query=query,
#                 limit=limit,
#                 sleep=sleep_time,
#                 sp_filter=sp_filter,
#                 results_type=results_type,
#                 proxies=proxies,
#             )

#             # Convert generator to a list to check its content
#             video_meta_list = list(video_meta_generator)
#             if not video_meta_list:
#                 logging.warning(f"No results found for query '{query}'. Skipping...")
#                 continue

#         except Exception as e:
#             logging.error(f"Failed to fetch metadata for query '{query}': {e}")
#             continue

#         items = []
#         logging.info("Fetching metadata...")
#         for video_meta in tqdm(
#             video_meta_list,
#             total=min(len(video_meta_list), limit),
#             desc=f"Processing {query}",
#         ):
#             try:
#                 # Extract metadata
#                 video_id = video_meta["videoId"].strip()
#                 title = video_meta["title"]["runs"][0]["text"].strip()
#                 channel = video_meta["longBylineText"]["runs"][0]["text"].strip()
#                 at_channel = video_meta["longBylineText"]["runs"][0][
#                     "navigationEndpoint"
#                 ]["browseEndpoint"]["canonicalBaseUrl"].strip()
#                 view_count = video_meta["viewCountText"]["simpleText"]

#                 # Add to item list
#                 items.append(
#                     {
#                         "query_string" : query,
#                         "video_id": video_id,
#                         "title": title,
#                         "channel": channel,
#                         "at_channel": at_channel,
#                         "view_count": view_count,
#                         "video_url": f"{video_base_url}{video_id}",
#                     }
#                 )

#                 # Append raw JSON for detailed analysis later
#                 all_jsons.append(video_meta)

#             except KeyError as e:
#                 logging.warning(f"KeyError encountered while processing a video: {e}")
#                 continue

#         # Convert to DataFrame and append
#         if items:
#             all_dataframes.append(pd.DataFrame(items))

#         # Sleep to avoid being blocked
#         sleep_interval = random.randint(3, 7)
#         logging.info(f"Sleeping for {sleep_interval} seconds...")
#         sleep(sleep_interval)

#     logging.info("Finished fetching metadata.")
#     return pd.DataFrame(all_dataframes), all_jsons

In [ ]:
# # Set up logging
# logging.basicConfig(
#     level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
# )


# def fetch_video_meta(
#     query: str,
#     limit: int,
#     sleep_min: int,
#     sleep_max: int,
#     sp_filter: str,
#     results_type: str,
#     proxies: list[dict[str, str]],
#     video_base_url: str,
#     file_path_csv: str,
#     file_path_json: str,
# ) -> tuple[pd.DataFrame, list[dict]]:
#     """
#     Fetch video metadata for a single query.
#     """
#     try:
#         logging.info(f"Searching YouTube for: {query}")

#         proxy = random.choice(proxies)

#         # Random sleep time for throttling
#         sleep_time = random.randint(sleep_min, sleep_max)
#         video_meta_generator = get_search_cc(
#             query=query,
#             limit=limit,
#             sleep=sleep_time,
#             sp_filter=sp_filter,
#             results_type=results_type,
#             proxies=proxy,
#         )

#         video_meta_list = list(video_meta_generator)
#         if not video_meta_list:
#             logging.warning(f"No results found for query '{query}'. Skipping...")
#             return pd.DataFrame(), []

#         items = []
#         logging.info("Fetching metadata...")
#         for video_meta in tqdm(
#             video_meta_list,
#             total=min(len(video_meta_list), limit),
#             desc=f"Processing {query}",
#         ):
#             try:
#                 # Extract metadata with fallback to None for missing keys
#                 video_id = video_meta.get("videoId", "").strip() or None

#                 title = (
#                     video_meta.get("title", {})
#                     .get("runs", [{}])[0]
#                     .get("text", "")
#                     .strip()
#                     or None
#                 )

#                 channel = (
#                     video_meta.get("longBylineText", {})
#                     .get("runs", [{}])[0]
#                     .get("text", "")
#                     .strip()
#                     or None
#                 )

#                 at_channel = (
#                     video_meta.get("longBylineText", {})
#                     .get("runs", [{}])[0]
#                     .get("navigationEndpoint", {})
#                     .get("browseEndpoint", {})
#                     .get("canonicalBaseUrl", "")
#                     .strip()
#                     or None
#                 )

#                 view_count = video_meta.get("viewCountText", {}).get("simpleText", None)
#                 # Add to item list
#                 items.append(
#                     {
#                         "query": query,
#                         "video_id": video_id,
#                         "title": title,
#                         "channel": channel,
#                         "at_channel": at_channel,
#                         "view_count": view_count,
#                         "video_url": f"{video_base_url}{video_id}",
#                     }
#                 )

#             except KeyError as e:
#                 logging.warning(
#                     f"KeyError encountered while processing query {query}: {e}"
#                 )
#                 continue

#         df = pd.DataFrame(items)

#         # save append df
#         try:
#             # Check if the file exists
#             with open(file_path_csv, "r"):
#                 # If the file exists, append without writing the header
#                 df.to_csv(file_path_csv, mode="a", header=False, index=False)
#         except FileNotFoundError:
#             # If the file does not exist, write with the header
#             df.to_csv(file_path_csv, mode="w", header=True, index=False)

#         # save append json
#         # Append data to the JSON file
#         try:
#             # Read existing data
#             with open(file_path_json, "r") as file:
#                 existing_data = json.load(file)
#         except FileNotFoundError:
#             # File does not exist, start with an empty list
#             existing_data = []

#         # Add the new data to the existing data
#         existing_data.extend(video_meta_list)

#         # Write the updated data back to the file
#         with open(file_path_json, "w") as file:
#             json.dump(existing_data, file, indent=4)

#         return df, video_meta_list

#     except Exception as e:
#         logging.error(f"Error fetching metadata for query '{query}': {e}")
#         return pd.DataFrame(), []


# def get_cc_video_meta(
#     queries: list[str],
#     limit: int,
#     sleep_min: int,
#     sleep_max: int,
#     sp_filter: str,
#     file_path_csv:str,
#     file_path_json:str,
#     results_type: str = "video",
#     proxies: list[dict[str, str]] = None,
#     max_workers: int = 4,
# ) -> tuple[pd.DataFrame, list[dict]]:
#     """
#     Retrieve video metadata for multiple queries using parallel processing.

#     Args:
#         queries (List[str]): List of search queries to fetch metadata for.
#         limit (int): Maximum number of results per query.
#         sleep_min (int): Minimum sleep time between requests.
#         sleep_max (int): Maximum sleep time between requests.
#         sp_filter (str): Filter parameter for YouTube search.
#         results_type (str, optional): Type of result to fetch (default is 'video').
#         proxies (Optional[Dict[str, str]], optional): Proxy configuration (default is None).
#         max_workers (int): Number of parallel workers to use.

#     Returns:
#         Tuple[List[pd.DataFrame], List[dict]]: A tuple containing:
#             - A list of Pandas DataFrames with metadata.
#             - A list of raw JSON responses.
#     """
#     video_base_url = "https://www.youtube.com/watch?v="
#     all_dataframes = []
#     all_jsons = []

#     # Parallel processing with ThreadPoolExecutor
#     with ThreadPoolExecutor(max_workers=max_workers) as executor:
#         futures = [
#             executor.submit(
#                 fetch_video_meta,
#                 query,
#                 limit,
#                 sleep_min,
#                 sleep_max,
#                 sp_filter,
#                 results_type,
#                 proxies,
#                 video_base_url,
#             )
#             for query in queries
#         ]

#         for future in as_completed(futures):
#             try:
#                 df, json_list = future.result()
#                 if not df.empty:
#                     all_dataframes.append(df)
#                     all_jsons.extend(json_list)
#             except Exception as e:
#                 logging.error(f"Error in processing: {e}")

#     logging.info("Finished fetching metadata.")
#     return pd.concat(all_dataframes), all_jsons

In [ ]:
# import threading

# # Set up logging
# logging.basicConfig(
#     level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
# )

# # Create locks for thread-safe file operations
# csv_lock = threading.Lock()
# json_lock = threading.Lock()


# def save_to_csv(df: pd.DataFrame, file_path_csv: str):
#     """
#     Save or append DataFrame to a CSV file in a thread-safe manner.
#     """
#     with csv_lock:  # Ensure only one thread writes to the file at a time
#         try:
#             # Check if the file exists
#             with open(file_path_csv, "r"):
#                 # Append without writing the header if the file exists
#                 df.to_csv(file_path_csv, mode="a", header=False, index=False)
#         except FileNotFoundError:
#             # Write with the header if the file does not exist
#             df.to_csv(file_path_csv, mode="w", header=True, index=False)


# def save_to_json(video_meta_list: list[dict], file_path_json: str):
#     """
#     Append data to a JSON file in a thread-safe manner.
#     """
#     with json_lock:  # Ensure only one thread writes to the file at a time
#         try:
#             # Read existing data
#             with open(file_path_json, "r") as file:
#                 existing_data = json.load(file)
#         except FileNotFoundError:
#             # Start with an empty list if the file does not exist
#             existing_data = []

#         # Add new data to the existing data
#         existing_data.extend(video_meta_list)

#         # Write the updated data back to the file
#         with open(file_path_json, "w") as file:
#             json.dump(existing_data, file, indent=4)


# def fetch_video_meta(
#     query: str,
#     limit: int,
#     sleep_min: int,
#     sleep_max: int,
#     sp_filter: str,
#     results_type: str,
#     proxies: list[dict[str, str]],
#     video_base_url: str,
#     file_path_csv: str,
#     file_path_json: str,
# ) -> tuple[pd.DataFrame, list[dict]]:
#     """
#     Fetch video metadata for a single query.
#     """
#     try:
#         logging.info(f"Searching YouTube for: {query}")

#         proxy = random.choice(proxies)

#         # Random sleep time for throttling
#         sleep_time = random.randint(sleep_min, sleep_max)
#         video_meta_generator = get_search_cc(
#             query=query,
#             limit=limit,
#             sleep=sleep_time,
#             sp_filter=sp_filter,
#             results_type=results_type,
#             proxies=proxy,
#         )

#         video_meta_list = list(video_meta_generator)
#         if not video_meta_list:
#             logging.warning(f"No results found for query '{query}'. Skipping...")
#             return pd.DataFrame(), []

#         items = []
#         logging.info("Fetching metadata...")
#         for video_meta in tqdm(
#             video_meta_list,
#             total=min(len(video_meta_list), limit),
#             desc=f"Processing {query}",
#         ):
#             try:
#                 # Extract metadata with fallback to None for missing keys
#                 video_id = video_meta.get("videoId", "").strip() or None

#                 title = (
#                     video_meta.get("title", {})
#                     .get("runs", [{}])[0]
#                     .get("text", "")
#                     .strip()
#                     or None
#                 )

#                 channel = (
#                     video_meta.get("longBylineText", {})
#                     .get("runs", [{}])[0]
#                     .get("text", "")
#                     .strip()
#                     or None
#                 )

#                 at_channel = (
#                     video_meta.get("longBylineText", {})
#                     .get("runs", [{}])[0]
#                     .get("navigationEndpoint", {})
#                     .get("browseEndpoint", {})
#                     .get("canonicalBaseUrl", "")
#                     .strip()
#                     or None
#                 )

#                 view_count = video_meta.get("viewCountText", {}).get("simpleText", None)
#                 # Add to item list
#                 items.append(
#                     {
#                         "query": query,
#                         "video_id": video_id,
#                         "title": title,
#                         "channel": channel,
#                         "at_channel": at_channel,
#                         "view_count": view_count,
#                         "video_url": f"{video_base_url}{video_id}",
#                     }
#                 )

#             except KeyError as e:
#                 logging.warning(
#                     f"KeyError encountered while processing query {query}: {e}"
#                 )
#                 continue

#         df = pd.DataFrame(items)

#         # Save data to CSV and JSON files
#         save_to_csv(df, file_path_csv)
#         save_to_json(video_meta_list, file_path_json)

#         return df, video_meta_list

#     except Exception as e:
#         logging.error(f"Error fetching metadata for query '{query}': {e}")
#         return pd.DataFrame(), []


# def get_cc_video_meta(
#     queries: list[str],
#     limit: int,
#     sleep_min: int,
#     sleep_max: int,
#     sp_filter: str,
#     file_path_csv:str,
#     file_path_json:str,
#     results_type: str = "video",
#     proxies: list[dict[str, str]] = None,
#     max_workers: int = 4,
# ) -> tuple[pd.DataFrame, list[dict]]:
#     """
#     Retrieve video metadata for multiple queries using parallel processing.

#     Args:
#         queries (List[str]): List of search queries to fetch metadata for.
#         limit (int): Maximum number of results per query.
#         sleep_min (int): Minimum sleep time between requests.
#         sleep_max (int): Maximum sleep time between requests.
#         sp_filter (str): Filter parameter for YouTube search.
#         results_type (str, optional): Type of result to fetch (default is 'video').
#         proxies (Optional[Dict[str, str]], optional): Proxy configuration (default is None).
#         max_workers (int): Number of parallel workers to use.

#     Returns:
#         Tuple[List[pd.DataFrame], List[dict]]: A tuple containing:
#             - A list of Pandas DataFrames with metadata.
#             - A list of raw JSON responses.
#     """
#     video_base_url = "https://www.youtube.com/watch?v="
#     all_dataframes = []
#     all_jsons = []

#     # Parallel processing with ThreadPoolExecutor
#     with ThreadPoolExecutor(max_workers=max_workers) as executor:
#         futures = [
#             executor.submit(
#                 fetch_video_meta,
#                 query,
#                 limit,
#                 sleep_min,
#                 sleep_max,
#                 sp_filter,
#                 results_type,
#                 proxies,
#                 video_base_url,
#                 file_path_csv,
#                 file_path_json,
#             )
#             for query in queries
#         ]

#         for future in as_completed(futures):
#             try:
#                 df, json_list = future.result()
#                 if not df.empty:
#                     all_dataframes.append(df)
#                     all_jsons.extend(json_list)
#             except Exception as e:
#                 logging.error(f"Error in processing: {e}")

#     logging.info("Finished fetching metadata.")
#     return pd.concat(all_dataframes), all_jsons

In [ ]:
limit = 1_000
sleep_min = 2
sleep_max = 16
sp_filter = "relevance"
workers = 128
file_path_csv = f"./data/channel_df_{today_string}.csv"
file_path_json = f"./data/channel_df_{today_string}.json"
print(file_path_csv, file_path_json)

In [ ]:
# random.choice(proxies)
# ถ้าจะรันใหม่อย่าลืมไปลบ csv, json ด้วยยยย
get_cc_video_meta(
    queries=channels,
    limit=limit,
    sleep_min=sleep_min,
    sleep_max=sleep_max,
    sp_filter=sp_filter,
    file_path_csv=file_path_csv,
    file_path_json=file_path_json,
    max_workers=workers,
    proxies=proxies,
)

In [ ]:
1

## Use youtube_transcript_api

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound

In [ ]:
proxies = [
    # {"http": "http://103.25.210.233:9191"},
    # {"http": "http://67.43.227.226:30373"},
    {"http": "http://116.202.121.34:3128"},
    {"http": "http://116.108.3.96:10017"},
    {"http": "http://160.86.242.23:8080"},
    # {"http": "http://35.209.198.222:80"},
    # {"http": "http://20.27.86.185:8080"},
    {"http": "http://31.47.58.37:80"},
    {"http": "http://213.218.255.99:80"},
    {"http": "http://72.10.160.94:8355"},
]

In [ ]:
df = pd.read_csv("./channel_cc_video.csv")
df.info()

In [ ]:
df.sample(5)

In [ ]:
video_id = "mbmckE6eJkQ"
lang_code = ["en"]
available_subtitle = YouTubeTranscriptApi.list_transcripts(video_id)
try:
    subtitles = available_subtitle.find_transcript(lang_code)
    print(subtitles.is_generated)
    print(subtitles.video_id)
except NoTranscriptFound:
    print(f"NO subtitle for lang code {lang_code}")

In [ ]:
# out = YouTubeTranscriptApi.get_transcript(video_id, languages=["th"])
# type(out)
# len(out)

In [ ]:
# video_ids = ["ZP163YC1_9E", "lHsneMSnjgk"]
# try:
#     out = YouTubeTranscriptApi.get_transcripts(video_ids, languages=lang_code)
# except NoTranscriptFound:
#     print("")

In [ ]:
# def get_subtitle_from_video_id(
#         video_id:str,
#         lang_code:list[str],
#         proxies:list[dict[str, str]],
# ):
#     proxy = random.choice(proxies)
#     available_subtitle = YouTubeTranscriptApi.list_transcripts(video_id, proxies=proxy)
#     try:
#         subtitles = available_subtitle.find_transcript(lang_code)
#         return {
#             "video_id" : subtitles.video_id,
#             "is_generate" : subtitles.is_generated,
#             "subtitle" : subtitles.fetch()
#         }
#     except NoTranscriptFound:
#         print(f"NO subtitle for lang code {lang_code}")
#         return None

# lang_code = ["th"]
# subtitle_list = []
# for id in tqdm(df["video_id"], total=len(df), desc="getting subtitle"):
#     subtitle_list.append(
#         get_subtitle_from_video_id(
#             video_id=id,
#             lang_code=lang_code,
#             proxies=proxies,
#         )
#     )
#     sleep(random.randint(3, 10))

In [ ]:
import threading

threading.active_count()

In [ ]:
# Function to fetch subtitles
def get_subtitle_from_video_id(
    video_id: str, lang_code: list[str], proxies: list[dict[str, str]]
) -> dict | None:
    """
    Fetch subtitles for a given YouTube video ID.
    """
    proxy = random.choice(proxies)  # Randomly select a proxy
    try:
        available_subtitle = YouTubeTranscriptApi.list_transcripts(
            video_id, proxies=proxy
        )
        subtitles = available_subtitle.find_transcript(lang_code)
        return {
            "video_id": subtitles.video_id,
            "is_generate": subtitles.is_generated,
            "subtitle": subtitles.fetch(),
        }
    except NoTranscriptFound:
        print(f"No subtitle for lang code {lang_code} in video {video_id}")
        print()
        return {
            "video_id": video_id,
            "is_generate": None,
            "subtitle": None,
        }
    except Exception as e:
        print(f"Error fetching subtitles for video {video_id}: {e}")
        print()
        return {
            "video_id": video_id,
            "is_generate": None,
            "subtitle": None,
        }


# Thread-safe function with throttling
def fetch_with_throttle(
    video_id: str,
    lang_code: list[str],
    proxies: list[dict[str, str]],
    sleep_min: int = 3,
    sleep_max: int = 10,
) -> dict | None:
    """
    Wrapper function to introduce throttling between requests.
    """
    result = get_subtitle_from_video_id(video_id, lang_code, proxies)
    sleep(random.randint(sleep_min, sleep_max))  # Random delay to avoid rate limits
    return result


max_workers = 8
lang_code = ["th"]
subtitle_list = []
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = [
        executor.submit(fetch_with_throttle, video_id, lang_code, proxies)
        for video_id in df["video_id"]
    ]
    for future in tqdm(
        as_completed(futures), total=len(futures), desc="Fetching subtitles"
    ):
        try:
            subtitle_list.append(future.result())
        except Exception as e:
            print(f"Error processing future: {e}")

# Save results to DataFrame
subtitle_df = pd.DataFrame(subtitle_list)

In [ ]:
subtitle_list[0]

In [ ]:
tmp = joblib.dump(
    pd.DataFrame(
        [
            i if i else {"video_id": None, "is_generate": None, "subtitle": None}
            for i in subtitle_list
        ]
    ),
    "tmp.joblib",
)

In [ ]:
tmp_df = pd.DataFrame(
    [
        i if i else {"video_id": None, "is_generate": None, "subtitle": None}
        for i in subtitle_list
    ]
)
# tmp_df = tmp_df.loc[tmp_df["is_generate"].eq(False)]
subs = tmp_df["subtitle"].to_list()

In [ ]:
tmp_df

In [ ]:
len(subs)

In [ ]:
tmp_df.head()

In [ ]:
subs[0][:10]

In [ ]:
text_list = []
end = 0
for sub in subs[:1]:
    text_line = ""
    for s in sub:
        text = s["text"]
        start = s["start"]
        duration = s["duration"]
        if min(start - end, 0) < 1:
            text_line += text
        else:
            text_line += "\n"
            text_line += text

        end = start + duration
    text_list.append(text_line)

In [ ]:
print(text_line)

## tmp